# Baseline submission (Ludovico)

**Goal:** a first, simple Kaggle submission, sent **before** any preprocessing or feature work.

**Why now**
- **Test the whole chain:** clean test → predict → threshold → `.csv` in the right format.
- **Check the validation:** if the Kaggle score is close to the CV estimate, the time CV + gap can be trusted for every later decision. The rules also penalise a large gap between the reported and the Kaggle performance.
- **Reference point:** every later model must beat this one.

**The model:** the baseline of `validation_ludovico`, unchanged. Median imputation + scaling + one-hot → logistic regression, with the threshold tuned on the time CV.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>1. Expected score from the validation</font>](#expected)
* [<font color='#E8800A'>2. Fit on all train and predict test</font>](#fit)
* [<font color='#E8800A'>3. Format checks</font>](#checks)
* [<font color='#E8800A'>4. Save, log and upload</font>](#save)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

`%run` executes `validation_ludovico.ipynb`, which itself runs `cleaning_ludovico.ipynb`. This gives back the clean data (`train`, `test`), the features `X`, the models, the time-CV splits and the out-of-fold predictions, so nothing is redefined here.

In [1]:
%%capture
%run ./validation_ludovico.ipynb

In [2]:
from src.paths import DATA_RAW
from src.submissions import check_submission, publish, record_kaggle_score, submit

MODEL_NAME = "logistic regression"
SCHEME = "time CV + gap"

# <font color='#E8800A'>1. Expected score from the validation</font> <a class="anchor" id="expected"></a>
[Back to TOC](#toc)

**Before predicting test, write down what we expect.** Once the Kaggle score is known, compare it with this.

- **Final threshold:** the one with the best F1 on the out-of-fold predictions of **all** folds. It uses all the validation data, so it is the best choice for the submission.
- **Honest estimate:** F1 on fold 3 (closest to test) with the threshold chosen on folds 1–2 only. This is the number to compare with Kaggle.

In [3]:
fold_scores = results[MODEL_NAME, SCHEME]
oof = oof_predictions[MODEL_NAME, SCHEME]
covered = oof.notna()

threshold, _ = best_threshold(y[covered], oof[covered])

last = schemes[SCHEME][-1][1]
honest_f1 = f1_score(y.iloc[last], oof.iloc[last] >= t_early)

expected = pd.Series({
    "ROC AUC (mean over folds)": fold_scores["ROC AUC"].mean(),
    "final threshold (all folds)": threshold,
    "F1 at final threshold, out-of-fold (optimistic)": f1_score(y[covered], oof[covered] >= threshold),
    "honest F1: threshold from folds 1-2, scored on fold 3": honest_f1,
    "F1 'always 1' on fold 3 (floor)": f1_score(y.iloc[last], np.ones(len(last))),
}, name="value")
expected.to_frame().style.format("{:.3f}")

,value
ROC AUC (mean over folds),0.778
final threshold (all folds),0.340
"F1 at final threshold, out-of-fold (optimistic)",0.750
"honest F1: threshold from folds 1-2, scored on fold 3",0.771
F1 'always 1' on fold 3 (floor),0.754


**Expected Kaggle F1 ≈ the honest estimate**, give or take a few hundredths. The public leaderboard uses only part of the 986 test rows, so it is noisy.

# <font color='#E8800A'>2. Fit on all train and predict test</font> <a class="anchor" id="fit"></a>
[Back to TOC](#toc)

**Steps**
1. Refit the pipeline on **all** clean train rows. More data than any validation fold, same preprocessing.
2. Predict the probability of a repeat purchase for every test row. Imputer, scaler and encoder use **what they learned on train**: test NaNs get train medians.
3. Apply the threshold: 1 if probability ≥ threshold, else 0. Kaggle receives labels, not probabilities, so **the threshold is our choice**.

In [4]:
final_model = clone(models[MODEL_NAME]).fit(X, y)

X_test = test.drop(columns=[c for c in NOT_FEATURES if c in test.columns])
assert list(X_test.columns) == list(X.columns), "test features differ from train features"

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= threshold).astype(int)

submission = pd.DataFrame({"ID": test.index, TARGET: test_pred})
submission.head()

,ID,repeat_purchase_90d
0,10000006,1
1,10000008,1
2,10000010,1
3,10000020,1
4,10000025,0


# <font color='#E8800A'>3. Format checks</font> <a class="anchor" id="checks"></a>
[Back to TOC](#toc)

**Compare with `sample_submission.csv`**: same columns, same IDs in the same order, one 0/1 label per row, no missing values. Also compare the share of predicted 1s with train and with the provided baseline.

In [5]:
sample = pd.read_csv(DATA_RAW / "sample_submission.csv")

check_submission(submission)  # columns, IDs in the right order, 0/1 labels: raises if anything is wrong

pd.Series({
    "rows": len(submission),
    "share of 1s - our predictions": submission[TARGET].mean(),
    "share of 1s - sample_submission baseline": sample[TARGET].mean(),
    "share of 1s - train (actual)": y.mean(),
}, name="value").to_frame().style.format("{:.3f}")

,value
rows,986.000
share of 1s - our predictions,0.796
share of 1s - sample_submission baseline,0.752
share of 1s - train (actual),0.560


**Reading the numbers:** with a threshold well below 0.5 the model predicts 1 more often than the 56% of positives in train. That is expected: F1 on class 1 rewards recall.

# <font color='#E8800A'>4. Save, log and upload</font> <a class="anchor" id="save"></a>
[Back to TOC](#toc)

Everything goes through `src/submissions.py`, the same for every group member (instructions at the top of `docs/submissions.md`):
- `submit` checks the format, assigns the **next free version number**, saves `submissions/ML_Group_41_VersionXX.csv` and adds the row to `docs/submissions.md` (date, author, notebook, model, CV F1, notes). Same predictions as an existing file → that version is reused, so re-running this notebook is safe.
- `publish` commits **only** that file and the log, merges teammates' pushes and pushes. Run it **before** uploading on Kaggle, so the version number is taken.

In [6]:
version = submit(
    submission,
    notebook="submission_baseline_ludovico",
    model="logistic regression (median impute + scaling + one-hot)",
    cv_f1=honest_f1,  # honest estimate: threshold from folds 1-2, scored on fold 3
    notes=(f"threshold {threshold:.2f} from {SCHEME}; CV ROC AUC {fold_scores['ROC AUC'].mean():.3f}; "
           f"{submission[TARGET].mean():.0%} predicted 1; no feature work"),
)

V01 saved as submissions/ML_Group_41_Version01.csv and logged:
  Version: 01 | Date: 2026-10-11 | Author: Ludovico Piras | Notebook / model: submission_baseline_ludovico · logistic regression (median impute + scaling + one-hot) | CV F1: 0.771 | Kaggle public F1: 0.77033 | Notes: threshold 0.34 from time CV + gap; CV ROC AUC 0.778; 80% predicted 1; no feature work
Next: publish(1) to reserve the version, then upload the file on Kaggle.


**Reserve the version on GitHub** (commit + push of the CSV and the log only). The number changes only if a teammate took it first: upload the file printed here.

In [ ]:
version = publish(version)

### Upload on Kaggle
1. Competition page → **Submit Prediction**, team `ML_Group_41`.
2. Upload the file printed above (`submissions/ML_Group_41_VersionXX.csv`).
3. Description, e.g.: *"V01 baseline: logistic regression, median imputation, threshold from time CV"*.

Then record the public score and share it:

In [ ]:
record_kaggle_score(version, 0.77033)
publish(version)

### How to read the Kaggle score
- **Close to the honest estimate** (within ~0.03) → the validation reproduces the test well, so later decisions can rely on the CV.
- **Much lower** → something differs between validation and test (drift, a leak in the validation). Investigate before building on it.
- **Do not tune anything on the public score.** It covers only part of the test, and the rules ask for decisions based on sound validation.